# Image Classification Transfer learning with YOLOv11
![yolo](https://cdn.prod.website-files.com/680a070c3b99253410dd3df5/680a070c3b99253410dd4791_67ed5670d7ecbda0527fe8b3_66f680814693dd5c3b60dfcb_YOLO11_Thumbnail.png)

In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 8.4 MB/s eta 0:00:00


***augment is a Boolean flag that tells the dataset whether to apply data augmentation or not.***

`Data augmentation means creating variations of your training images by applying random changes.`

In [2]:
import torch
import torchvision.transforms as T

from ultralytics import YOLO
from ultralytics.data.dataset import ClassificationDataset

from ultralytics.models.yolo.classify import (
    ClassificationTrainer,
    ClassificationValidator,
    ClassificationPredictor
)


class CustomizedDataset(ClassificationDataset):
    """Customized dataset for image classification with custom augmentation."""

    def __init__(
        self,
        root: str,
        args,
        augment: bool = True,
        prefix: str = ""
    ):

        super().__init__(root, args, augment, prefix)

        # Training transformations
        train_transforms = T.Compose([
            T.Resize((args.imgsz, args.imgsz)),

            T.RandomHorizontalFlip(
                p=args.fliplr
            ),

            T.RandomVerticalFlip(
                p=args.flipud
            ),

            T.RandomAffine(
                degrees=10,
                interpolation=T.InterpolationMode.BILINEAR
            ),

            T.ColorJitter(
                brightness=args.hsv_v,
                contrast=args.hsv_v,
                saturation=args.hsv_s,
                hue=args.hsv_h
            ),

            T.ToTensor(),

            T.Normalize(
                mean=(0.5, 0.5, 0.5),
                std=(0.5, 0.5, 0.5)
            ),

            T.RandomErasing(
                p=args.erasing,
                inplace=True
            )
        ])

        # Validation transformations
        val_transform = T.Compose([
            T.Resize((args.imgsz, args.imgsz)),

            T.ToTensor(),

            T.Normalize(
                mean=(0.5, 0.5, 0.5),
                std=(0.5, 0.5, 0.5)
            )
        ])

        self.torch_transforms = (
            train_transforms if augment else val_transform
        )


class CustomizedTrainer(ClassificationTrainer):
    """Customized trainer for YOLO classification."""

    def build_dataset(
        self,
        img_path: str,
        mode: str = "train",
        batch=None
    ):
        """Build customized dataset."""

        return CustomizedDataset(
            root=img_path,
            args=self.args,
            augment=(mode == "train"),
            prefix=mode
        )


class CustomizedValidator(ClassificationValidator):
    """Customized validator for YOLO classification."""

    def build_dataset(
        self,
        img_path: str,
        mode: str = "val",
        batch=None
    ):
        """Build customized validation dataset."""

        return CustomizedDataset(
            root=img_path,
            args=self.args,
            augment=False,
            prefix=mode
        )

***`DOWNLOAD DATASETS`***

In [3]:
import gdown

# Google Drive file ID
file_id = "1TCU1nqgIe1R_dW6LTkRxlufHlCCazyJl"
# Download destination filename
output = "myfile.zip"

# Download the file
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1TCU1nqgIe1R_dW6LTkRxlufHlCCazyJl
From (redirected): https://drive.google.com/uc?id=1TCU1nqgIe1R_dW6LTkRxlufHlCCazyJl&confirm=t&uuid=9e5d6082-c3c0-411b-b2da-a5d9e08b3ff1
To: /content/myfile.zip
100%|██████████| 63.9M/63.9M [00:01<00:00, 45.9MB/s]


'myfile.zip'

In [4]:
import zipfile

with zipfile.ZipFile("myfile.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

***Custom Training or Transfer learning***

In [5]:
from ultralytics import YOLO

model = YOLO('yolo11n-cls.pt')

model.train(data='/content/dataset/train',trainer=CustomizedTrainer,epochs=10,
            imgsz=224,batch = 64)

Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/train, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=Non

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bedbe10ce60>
curves: []
curves_results: []
fitness: 0.986328125
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.97265625, 'metrics/accuracy_top5': 1.0, 'fitness': 0.986328125}
save_dir: PosixPath('/content/runs/classify/train')
speed: {'preprocess': 0.08499476562207064, 'inference': 0.2546443437481116, 'loss': 8.458984268600034e-05, 'postprocess': 0.0001567929679424651}
top1: 0.97265625
top5: 1.0

In [6]:
metrics = model.val(data='/content/dataset/valid',
                    validator=CustomizedValidator,
                    imgsz = 224,
                    batch=64)

Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 3.2 GFLOPs
WARNING ⚠️ Dataset 'split=train' not found at /content/dataset/valid/train
Found 364 images in subdirectories. Attempting to split...
Splitting /content/dataset/valid (2 classes, 364 images) into 80% train, 20% val...
Split complete in /content/dataset/valid_split ✅
train: /content/dataset/valid_split/train... found 290 images in 2 classes ✅ 
val: /content/dataset/valid_split/val... found 74 images in 2 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 879.3±385.0 MB/s, size: 37.4 KB)
val: Scanning /content/dataset/valid_split/val... 74 images, 0 corrupt: 100% ━━━━━━━━━━━━ 74/74 5.5Kit/s 0.0s
val: New cache created: /content/dataset/valid_split/val.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 1.8it/s 1.1s
                   all      0.973          1
Speed: 1.4ms pre

In [8]:
metrics.top1

0.9729729890823364

In [9]:
metrics.top5

1.0

In [11]:
model=YOLO('/content/runs/classify/train/weights/best.pt')
results = model(
    "/content/dataset/test/daisy/12193032636_b50ae7db35_n_jpg.rf.e6c4eeb71c56e793a0d85f6d979dbe20.jpg"
)
results


image 1/1 /content/dataset/test/daisy/12193032636_b50ae7db35_n_jpg.rf.e6c4eeb71c56e793a0d85f6d979dbe20.jpg: 224x224 daisy 1.00, dandelion 0.00, 39.5ms
Speed: 28.7ms preprocess, 39.5ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)


[ultralytics.engine.results.Results object with attributes:
 
 boxes: None
 depth: None
 keypoints: None
 masks: None
 names: {0: 'daisy', 1: 'dandelion'}
 obb: None
 orig_img: array([[[ 2, 13,  5],
         [ 2, 13,  5],
         [ 2, 13,  5],
         ...,
         [ 3, 22, 13],
         [ 3, 22, 13],
         [ 3, 22, 13]],
 
        [[ 2, 13,  5],
         [ 2, 13,  5],
         [ 2, 13,  5],
         ...,
         [ 3, 22, 13],
         [ 3, 22, 13],
         [ 3, 22, 13]],
 
        [[ 2, 13,  5],
         [ 2, 13,  5],
         [ 2, 13,  5],
         ...,
         [ 3, 22, 13],
         [ 3, 22, 13],
         [ 3, 22, 13]],
 
        ...,
 
        [[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [ 0, 31, 22],
         [ 0, 31, 22],
         [ 0, 31, 22]],
 
        [[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [ 0, 31, 22],
         [ 0, 31, 22],
         [ 0, 31, 22]],
 
        [[ 0,  0,  0],
         [ 

In [12]:
# Predict with the model
results = model("/content/dataset/test/dandelion/16159487_3a6615a565_n_jpg.rf.6d473a1fe680a3e930f3ff28464c46a9.jpg")


image 1/1 /content/dataset/test/dandelion/16159487_3a6615a565_n_jpg.rf.6d473a1fe680a3e930f3ff28464c46a9.jpg: 224x224 dandelion 1.00, daisy 0.00, 4.7ms
Speed: 3.8ms preprocess, 4.7ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
